# Initial Device Performance Prediction (JV Default PCE)

## 1. Introduction
Predicting the initial Power Conversion Efficiency ($PCE$) of a device based on its architecture and material composition is vital for rapid prototyping. This notebook focuses on the **JV Default PCE** prediction.

## 2. Methodology
*   **Target:** Initial efficiency as measured by J-V characteristic sweeps.
*   **Features:** Layer sequences (ETL/HTL), device architecture (nip/pin), and perovskite band gap.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_absolute_error

# EXPLICIT WHITE THEME
import matplotlib as mpl
mpl.rcParams['figure.facecolor'] = 'white'
mpl.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['savefig.facecolor'] = 'white'
sns.set_theme(style="white", palette="Set2")

# Load device performance data
df_jv = pd.read_parquet('../data/solar_panels.parquet').dropna(subset=['JV_default_PCE'])
model_pce = joblib.load('../ml_models/pce_jv_model.joblib')

print(f"Dataset contains {len(df_jv)} unique device configurations.")

## 3. Architecture Impact Analysis
Does the choice between **n-i-p** and **p-i-n** architecture fundamentally change the efficiency ceiling?

In [ ]:
plt.figure(figsize=(10, 6), facecolor='white')
ax = sns.boxplot(data=df_jv, x='Cell_architecture', y='JV_default_PCE', palette='Set2')
ax.set_facecolor("white")
plt.title("Efficiency Distribution by Device Architecture", fontsize=14)
plt.xlabel("Architecture", fontsize=12)
plt.ylabel("Initial PCE (%)", fontsize=12)
plt.show()

## 4. Prediction Accuracy
Visualizing the model's ability to predict device efficiency across different stacks.

In [ ]:
features = [
    'A_1', 'A_2', 'A_3', 'B_1', 'C_1', 'C_2',
    'en_A', 'en_B', 'en_C', 'mass_A', 'mass_B', 'mass_C',
    'tolerance_factor', 'octahedral_factor', 'is_2d', 'dimension', 'space_group',
    'Cell_architecture', 'ETL_stack_sequence', 'HTL_stack_sequence', 'Backcontact_stack_sequence', 
    'Cell_area_measured', 'Perovskite_band_gap', 
    'composition_inorganic'
]

y_true = df_jv['JV_default_PCE']
y_pred = model_pce.predict(df_jv[features])

plt.figure(figsize=(6, 6), facecolor='white')
plt.scatter(y_true, y_pred, alpha=0.4, color='#16a085')
plt.plot([0, 30], [0, 30], 'k--', lw=1.5)
plt.gca().set_facecolor("white")
plt.title(f"Device Performance Parity (R² = {r2_score(y_true, y_pred):.3f})", fontsize=14)
plt.xlabel("Experimental PCE (%)", fontsize=12)
plt.ylabel("Predicted PCE (%)", fontsize=12)
plt.xlim(0, 26)
plt.ylim(0, 26)
plt.show()

## 5. Conclusion
The XGBoost model captures the complex interplay between material composition and stack sequence. This model serves as the primary evaluation engine in our **Inverse Design** pipeline, enabling the search for high-efficiency device architectures.